# Chapter 9 — A blind search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/09_blind_search.ipynb)

**Goal.** Run the search of Chapter 5, unchanged, on a region of the SDSS sky we have not looked at: RA 165°–181°,
Dec −3° to +8° (~180 deg² observed, ~140 deg² searched). Calibrate the threshold on null realisations of *this* region, list the candidates, and
only then compare them with the catalogue of known satellites.

**Rules of the game.** Everything — the CMD masks, kernels, distance grid, threshold definition and matching radius —
was fixed in Chapters 5 and 8 before this notebook was written. Do not change them after looking at the result; if you
do, the calibration no longer applies.

**You will learn**
1. to handle a survey footprint (a coverage mask) in a large-area search;
2. to calibrate a detection threshold for a whole region;
3. to extract candidates and write a short candidate report.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, maximum_filter          # Gaussian kernels; local maxima of the map
from scipy import special, stats                                  # incomplete gamma (Poisson tail), normal quantiles

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts of survey catalogues, attached to a GitHub release of the tutorial.
# MANIFEST.json lists, for each file, the SQL that made it and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)


def tangent(ra, dec, ra0, dec0):
    """Gnomonic (tangent-plane) projection about (ra0, dec0). Returns (xi, eta) in degrees: xi towards east, eta north."""
    # Flat coordinates around the target: over a few degrees the distortion is small, and unlike (ra, dec)
    # they do not stretch the sky by 1/cos(dec), so circles on the sky stay circles in (xi, eta).
    ra, dec, ra0, dec0 = map(np.radians, (ra, dec, ra0, dec0))
    cosc = np.sin(dec0) * np.sin(dec) + np.cos(dec0) * np.cos(dec) * np.cos(ra - ra0)   # cos(angular distance)
    xi = np.cos(dec) * np.sin(ra - ra0) / cosc
    eta = (np.cos(dec0) * np.sin(dec) - np.sin(dec0) * np.cos(dec) * np.cos(ra - ra0)) / cosc
    return np.degrees(xi), np.degrees(eta)


def untangent(xi, eta, ra0, dec0):
    """Inverse gnomonic projection: (xi, eta) in degrees -> (ra, dec) [deg], ra in [0, 360)."""
    xi, eta, ra0, dec0 = np.radians(xi), np.radians(eta), np.radians(ra0), np.radians(dec0)
    rho = np.hypot(xi, eta)                                       # distance from the tangent point on the plane
    c = np.arctan(rho)                                            # angular distance on the sky
    # at the tangent point itself (rho = 0) the second term is 0; np.maximum avoids a division by zero there
    dec = np.arcsin(np.cos(c) * np.sin(dec0) + np.where(rho > 0, eta * np.sin(c) * np.cos(dec0) / np.maximum(rho, 1e-300), 0))
    ra = ra0 + np.arctan2(xi * np.sin(c), rho * np.cos(dec0) * np.cos(c) - eta * np.sin(dec0) * np.sin(c))
    return np.degrees(ra) % 360, np.degrees(dec)


iso = load('isochrones_sdss')               # PARSEC 1.2S, SDSS ugriz: log age 9.9-10.1, [Fe/H] -2.1...-1.0

## 9.1 The frozen search

Copied from Chapter 5 without changes, except that it keeps, for every pixel, only the maximum of $S$ over distance
moduli and kernels (and which ones gave it): the region is ~1000 × 700 pixels, too large to keep all 44 maps.

In [ ]:
# CMD grid as in Chapter 5: (g-r)_0 from -0.4 to 1.2 in 0.04-mag bins, g_0 from 16 to 22.5 in 0.1-mag bins;
# cc, mm: bin centres; CC, MM: their 2-d grid, shape (40, 65) = (colour, magnitude)
cb, mb = np.arange(-0.4, 1.21, 0.04), np.arange(16, 22.51, 0.1)
cc, mm = 0.5 * (cb[1:] + cb[:-1]), 0.5 * (mb[1:] + mb[:-1])
CC, MM = np.meshgrid(cc, mm, indexing='ij')
# search isochrone: 12.6 Gyr (log age 10.1), [Fe/H] = -2.1, the oldest and most metal-poor of the grid (Chapter 2);
# evolutionary stages up to 7 (early AGB), ordered by initial mass
I = iso[np.isclose(iso.logage, 10.1) & np.isclose(iso.feh, -2.1) & (iso.stage <= 7)].sort_values('m_ini')
_c, _g = (I.g - I.r).to_numpy(), I.g.to_numpy()                   # absolute colour and absolute g magnitude
# 3000 points uniform in arc length along the track in the CMD, each weighted by the stars it represents (the
# increment of the cumulative IMF int_imf), as in Chapter 2: ISO_C, ISO_G [mag], ISO_W, each of shape (3000,)
_arc = np.concatenate([[0], np.cumsum(np.hypot(np.diff(_c), np.diff(_g)))])
_s = np.linspace(0, _arc[-1], 3000)
ISO_C, ISO_G, ISO_W = np.interp(_s, _arc, _c), np.interp(_s, _arc, _g), np.gradient(np.interp(_s, _arc, I.int_imf))
MUS = np.arange(16.5, 21.51, 0.5)          # trial distance moduli [mag]: 11 values, 20 to 200 kpc
SIGMAS = np.array([1, 2, 4, 8])            # Gaussian kernel widths [pixels = arcmin]


def satellite_cmd(mu, err=lambda m: 0.02 + 0.08 * np.exp((m - 22) / 1.0)):
    """Satellite CMD density s at distance modulus mu [mag]: shape (40, 65), sums to 1.

    err(m): one-band photometric error [mag] at magnitude m; it rises from 0.02 mag at the bright end to 0.10 mag at
    g = 22 (the median SDSS error there is ~0.07 mag).
    """
    pg = ISO_G + mu                                               # apparent g of the isochrone points
    # Gaussian blur: colour sqrt(2) x the one-band error plus an intrinsic 0.03 mag; magnitude: the error plus 0.1 mag
    # for the spread in distance and in horizontal-branch luminosity (Chapter 2)
    sc, sm = np.hypot(np.sqrt(2) * err(pg), 0.03), np.hypot(err(pg), 0.1)
    # weighted sum of the 3000 2-d Gaussians on the grid: (3000,) and (3000, 40, 65) -> (40, 65)
    S = np.einsum('k,kij->ij', ISO_W, np.exp(-0.5 * ((CC - ISO_C[:, None, None]) / sc[:, None, None])**2
                                             - 0.5 * ((MM - pg[:, None, None]) / sm[:, None, None])**2) / sc[:, None, None] / sm[:, None, None])
    return S / S.sum()


def cmd_index(c, m):
    """Flat CMD-cell index i * len(mm) + j of every star with colour c and magnitude m, or -1 off the grid."""
    i, j = np.digitize(c, cb) - 1, np.digitize(m, mb) - 1
    ok = (i >= 0) & (i < len(cc)) & (j >= 0) & (j < len(mm))
    return np.where(ok, i * len(mm) + j, -1)


def cmd_mask(S, B, keep=0.5):
    """Boolean CMD mask (Chapter 5): the cells of highest s/b that maximise sum(s) / sqrt(sum(b)).

    S, B: satellite and field CMD densities on the same grid (each sums to 1). keep: the mask must hold at least this
    fraction of the satellite's stars (half).
    """
    order = np.argsort((S / B).ravel())[::-1]                     # cells ranked by s/b, best first
    cs, cbk = np.cumsum(S.ravel()[order]), np.cumsum(B.ravel()[order])   # signal and background kept by the top k cells
    M = np.zeros(S.size, bool)
    # best k: maximum of cs / sqrt(cbk) among the k with cs >= keep (the others are set to 0 so that argmax skips them)
    M[order[:np.argmax(np.where(cs >= keep, cs / np.sqrt(cbk), 0)) + 1]] = True
    return M.reshape(S.shape)


def search_max(x, y, w, mask, extent, pix=1 / 60, sig_bg=30):
    """Maximum of the Chapter 5 significance over (mu, sigma) for every pixel, and the indices of the maximum.

    x, y: star positions [deg]; w: (len(MUS), n_stars) mask weights (0/1); mask: boolean footprint on the pixel grid;
    extent: (xmin, xmax, ymin, ymax) [deg]; pix: pixel size [deg] (1'); sig_bg: background kernel [pixels] (30').
    Returns best, shape mask.shape (0 where there is no excess or the pixel is not usable), and arg, shape
    mask.shape + (2,): the indices into MUS and SIGMAS that gave the maximum.
    """
    # pixel edges [deg]
    xe, ye = np.arange(extent[0], extent[1] + pix / 2, pix), np.arange(extent[2], extent[3] + pix / 2, pix)
    M = mask.astype(float)
    norm_bg = gaussian_filter(M, sig_bg)                          # fraction of the 30' kernel inside the footprint
    best = np.zeros(mask.shape); arg = np.zeros(mask.shape + (2,), int)
    norms = {s: gaussian_filter(M, s) for s in SIGMAS}            # the footprint is the same for every mu: compute once
    for a, wa in enumerate(w):                                    # one trial distance modulus at a time
        W = np.histogram2d(x, y, [xe, ye], weights=wa)[0]         # masked stars per pixel
        Wb = gaussian_filter(W, sig_bg) / np.maximum(norm_bg, 1e-9)   # background [stars per pixel], edge-corrected
        for b, s in enumerate(SIGMAS):
            norm = norms[s]                                       # fraction of the signal kernel inside the footprint
            area = 4 * np.pi * s**2 * norm                        # effective area 1 / sum(K^2) = 4 pi sigma^2 [pixels]
            n = gaussian_filter(W, s) / np.maximum(norm, 1e-9) * area   # effective count n = A L
            lam = np.maximum(Wb * area, 1e-9)                     # its expectation from the background
            # Gaussian-equivalent significance of the Poisson tail P(N >= n | lam) = gammainc(n, lam); the clip keeps
            # isf finite for extreme peaks
            S = stats.norm.isf(np.clip(special.gammainc(np.maximum(n, 1e-9), lam), 1e-300, 1))
            # excesses inside the footprint where at least half the kernel is observed; 0 elsewhere (not NaN, so that
            # the running maximum is defined everywhere)
            S = np.where(mask & (norm > 0.5) & (n > lam), S, 0.0)
            better = S > best                                     # update the running maximum and its (mu, sigma)
            best = np.where(better, S, best)
            arg[better] = (a, b)
    return best, arg

## 9.2 The region and its footprint

We project the region onto a tangent plane centred at (173°, 2.5°) with 1′ pixels. A pixel is "observed" if its 0.1°
coverage cell holds more than 20 SDSS objects. Pixels where less than 95% of the 30′ background kernel falls on observed sky (within ~0.8° of a straight footprint edge) are not searched, so that the
background kernel stays inside the data.

In [ ]:
RA0, DEC0 = 173.0, 2.5                          # tangent point [deg]: centre of the region RA 165-181, Dec -3...+8
cat = load('holdout_sdss')                      # SDSS point sources, -0.3 < g-r < 1.2, g < 22.5 (as pal5_sdss)
cov = load('holdout_coverage')                  # SDSS object counts n in 0.1 x 0.1 deg cells of (ra, dec): ira, idec
x, y = tangent(cat.ra.to_numpy(), cat.dec.to_numpy(), RA0, DEC0)  # tangent-plane coordinates [deg]
g0 = (cat.psfmag_g - cat.extinction_g).to_numpy()                 # extinction-corrected g_0 and (g-r)_0 [mag]
c0 = g0 - (cat.psfmag_r - cat.extinction_r).to_numpy()
extent = (np.floor(x.min()), np.ceil(x.max()), np.floor(y.min()), np.ceil(y.max()))   # map limits [deg], whole degrees
pix = 1 / 60                                    # 1' pixels, as in Chapter 5
px = np.arange(extent[0], extent[1], pix) + pix / 2                # pixel centres [deg]
py = np.arange(extent[2], extent[3], pix) + pix / 2
PX, PY = np.meshgrid(px, py, indexing='ij')                       # pixel grid, shape (1080, 720)
pra, pdec = untangent(PX, PY, RA0, DEC0)                          # (ra, dec) [deg] of every pixel centre
# Footprint: a pixel is observed if it lies inside the RA/Dec box of the extraction and its 0.1-deg coverage cell
# (index floor(ra / 0.1), floor(dec / 0.1)) holds more than 20 SDSS objects, as in Chapter 7.
observed_cells = set(zip(cov.ira[cov.n > 20], cov.idec[cov.n > 20]))
inbox = (pra >= 165) & (pra < 181) & (pdec >= -3) & (pdec < 8)
mask = inbox & np.array([(i, j) in observed_cells for i, j in zip(np.floor(pra.ravel() / 0.1).astype(int),
                                                               np.floor(pdec.ravel() / 0.1).astype(int))]).reshape(PX.shape)
# searched pixels: more than 95% of the 30' background kernel falls on observed pixels (about 0.8 deg, 1.645 kernel
# widths, from a straight edge)
searched = mask & (gaussian_filter(mask.astype(float), 30) > 0.95)
print(f'{len(cat):,} stars; observed area {mask.sum() * pix**2:.0f} deg²; searched area {searched.sum() * pix**2:.0f} deg²')

# CMD masks for the 11 trial distance moduli, with the field CMD b of the whole region (smoothed by one cell and floored
# at one star per cell, Chapter 4)
H = gaussian_filter(np.histogram2d(c0, g0, [cb, mb])[0], 1.0)
B = np.maximum(H, 1.0) / np.maximum(H, 1.0).sum()
masks = np.array([cmd_mask(satellite_cmd(mu), B).ravel() for mu in MUS])   # (11, 2600) boolean
k = cmd_index(c0, g0)
w = np.where(k >= 0, masks[:, np.maximum(k, 0)], False).astype(float)      # (11, n_stars): 1 if in the mask of that mu

## 9.3 Calibration on null realisations of this region

Each null realisation keeps every star's CMD position and places the stars uniformly over the observed pixels. We run
30 nulls (each takes a few seconds) and set the threshold at the largest null maximum, which corresponds to roughly a
1/30 chance per region of one false candidate above it. We also record the threshold that 95% of nulls stay below.

In [ ]:
rng = np.random.default_rng(9)                  # fixed seed: the same null realisations in every run
obs_pix = np.flatnonzero(mask)                                    # flat indices of the observed pixels
null_max = []
for t in range(30):                             # 30 null realisations (Section 9.3)
    # each star goes to a random observed pixel (uniform over the footprint), at a uniform position inside it
    p = rng.choice(obs_pix, len(x))
    nx = PX.ravel()[p] + rng.uniform(-pix / 2, pix / 2, len(x))
    ny = PY.ravel()[p] + rng.uniform(-pix / 2, pix / 2, len(x))
    Sn, _ = search_max(nx, ny, w, mask, extent)                   # same weights w: every star keeps its CMD position
    null_max.append(Sn[searched].max())                           # maximum over the searched area, all mu and kernels
null_max = np.array(null_max)
THRESHOLD = null_max.max()                      # the largest of 30 null maxima: ~1/30 chance per region of one false
                                                # candidate above it
print(f'null maxima over the region: median {np.median(null_max):.2f}, 95th percentile {np.percentile(null_max, 95):.2f}, '
      f'max {THRESHOLD:.2f} -> threshold')

## 9.4 The search

In [ ]:
S, arg = search_max(x, y, w, mask, extent)                        # the real data, searched once
S = np.where(searched, S, 0)                                      # keep only the searched area
peaks = (S == maximum_filter(S, size=15)) & (S > THRESHOLD)       # local maxima, at least 7' apart
ii, jj = np.nonzero(peaks)
# candidate table: S, position [deg], and the distance modulus and kernel width [arcmin] of the maximum, by S
cand = pd.DataFrame(dict(S=S[ii, jj], ra=pra[ii, jj], dec=pdec[ii, jj], mu=MUS[arg[ii, jj, 0]],
                         sigma_arcmin=SIGMAS[arg[ii, jj, 1]])).sort_values('S', ascending=False, ignore_index=True)
print(f'{len(cand)} candidates above S = {THRESHOLD:.2f}')
print(cand.round(2).to_string())

# map of S over the observed area (unobserved pixels NaN, i.e. blank), colour scale 0-8; white contour: the searched
# area; cyan circles and numbers: the candidates. The xi axis is inverted so that east is to the left, as on the sky.
fig, ax = plt.subplots(figsize=(13, 7))
im = ax.imshow(np.where(mask, S, np.nan).T, origin='lower', extent=extent, cmap='magma', vmin=0, vmax=8)
ax.contour(PX, PY, searched, [0.5], colors='w', linewidths=0.5)
for n, c in cand.iterrows():
    cx, cy = tangent(c.ra, c.dec, RA0, DEC0)
    ax.plot(cx, cy, 'o', mfc='none', mec='cyan', ms=14)
    ax.text(cx, cy + 0.25, str(n), color='cyan', ha='center')
ax.invert_xaxis()
ax.set(xlabel=r'$\xi$ [deg] (centre RA 173°, Dec 2.5°)', ylabel=r'$\eta$ [deg]', title='maximum S over distance and size')
plt.colorbar(im, label='S')
plt.tight_layout()

**Figure 9.1.** The search map of the held-out region RA 165°–181°, Dec −3° to +8° (`holdout_sdss`: 418,496 SDSS point
sources, extinction-corrected), on a tangent plane centred at (173°, 2.5°) with 1′ pixels; ξ, η in degrees, east to
the left. Colour (0 to 8): for every pixel, the largest significance S over the 11 distance moduli (μ = 16.5–21.5) and
the 4 kernels (1′–8′). Blank (white): outside the extraction box or in 0.1° coverage cells with ≤ 20 SDSS objects (the
small white rectangles are such holes). Black margin: observed, but not searched (S set to 0). The thin white contour
bounds the searched area (136 deg² of the 178 deg² observed), where more than 95% of the 30′ background kernel falls
on observed pixels. Cyan circles mark the 16 local maxima (at least 7′ apart) above the threshold S = 6.03, the
largest of 30 null maxima, numbered 0–15 in order of decreasing S (13.7 for #0, 6.1 for #15); 13 of the 16 peak with
the 1′ kernel, and 11 at μ ≥ 21. A uniform null predicts about one candidate in 30 such regions above this threshold;
we find 16, which the candidate reports below have to explain.

## 9.5 Candidate reports

$S$ says only that a pixel holds more stars in the CMD mask than the smooth background predicts. Before anything is
called a satellite, each candidate gets a **report card** that shows the evidence behind its peak.

**Top row: four maps of 30′ × 30′ around the peak** (east to the left; the blue circle is the aperture used below).
1. *All stars* (no CMD selection), smoothed with a 1′ Gaussian: is the overdensity there without the mask, and is the
   field clean — no footprint edge, stripe, hole or halo of a bright star?
2. *Stars in the CMD mask* at the candidate's μ — the stars that produced $S$: how large is the overdensity?
3. *S*, the search map (maximum over μ and σ, as in Figure 9.1): an isolated peak, or part of a ridge or an edge?
4. *Galaxies* (SDSS `type = 3`, $r < 21$), with bright galaxies ($r_0 < 17$) as red circles: is there a galaxy group
   or cluster, or a large galaxy that the pipeline may have split into star-like pieces?

**Bottom row: Hess diagrams** (0.1 mag in colour × 0.5 mag in $g_0$) of all stars inside the aperture, of radius
max(2σ, 3′), and of the 30′–60′ annulus, and their difference, aperture − α × annulus, where α is the ratio of the
aperture area to the observed area of the annulus. Without a stellar system the difference scatters around zero. The
orange line is the search isochrone at the candidate's μ, the green contour its CMD mask; the title of the card gives
the excess of masked stars in the aperture with its Poisson error.

**What to look for.**

| | real satellite | shredded galaxy | clustered faint galaxies | artefact |
|---|---|---|---|---|
| maps | resolved overdensity, size set by $r_h$ and the distance | knot of ≲ 1′ on a bright galaxy | knot of point sources, often near a galaxy overdensity | edges, stripes or holes in the all-star map |
| difference Hess | follows one isochrone: main sequence and turn-off, giant branch, horizontal branch | blue sources, bluer than the turn-off, no giants | faint sources at the magnitude limit, scattered in colour | — |

Two cautions. At $\mu \gtrsim 21$ (≳ 160 kpc) SDSS reaches only the horizontal branch and the brightest giants, so a
distant satellite shows only a few blue horizontal-branch stars and giants; a handful of faint blue sources cannot then
be told from faint blue galaxies, and size and deeper data (Chapter 12) have to decide. And the galaxy map is a warning,
not a verdict: galaxies are clustered everywhere, so a few galaxies near a candidate prove nothing on their own.

In [ ]:
def report_card(R, cx, cy, a, sig, label, half=15, r_off=(30, 60)):
    """Report card of a candidate at (cx, cy) [deg] of region R, for distance-modulus index a and kernel sig [arcmin].

    R: dict of the region with star positions x, y [deg], (g-r)_0 c0 and g_0 g0 [mag], mask weights w (len(MUS), n_stars),
    the search map S on the pixel grid PX, PY (1' pixels, map limits extent [deg]), the footprint mask, the CMD masks
    (len(MUS), 40 * 65), galaxy positions xg, yg [deg] and extinction-corrected galaxy magnitudes rg.
    Top row: maps of 2 half x 2 half arcmin around the candidate. Bottom row: Hess diagrams of an aperture of radius
    max(2 sig, 3') and of the annulus r_off [arcmin], and their difference with the annulus scaled by the area ratio.
    Returns the number of excess stars in the CMD mask and its Poisson error.
    """
    mu = MUS[a]
    dx, dy = (R['x'] - cx) * 60, (R['y'] - cy) * 60               # star offsets from the candidate [arcmin]
    gx, gy = (R['xg'] - cx) * 60, (R['yg'] - cy) * 60             # galaxy offsets [arcmin]
    sel = R['w'][a] > 0                                           # stars in the CMD mask of the candidate's mu
    r_on = max(2 * sig, 3)                                        # aperture radius [arcmin]
    e = np.arange(-half, half + 0.5, 1.0)                         # 1' map pixels
    # density per arcmin^2 on 1' pixels, smoothed with a 1' Gaussian
    dens = lambda u, v: gaussian_filter(np.histogram2d(u, v, [e, e])[0], 1.0)
    fig = plt.figure(figsize=(17, 8.6))
    gs = fig.add_gridspec(2, 12, hspace=0.38, wspace=2.2)
    top = [fig.add_subplot(gs[0, 3 * k:3 * k + 3]) for k in range(4)]
    bot = [fig.add_subplot(gs[1, 4 * k:4 * k + 4]) for k in range(3)]
    # top row: all stars, masked stars, the search map S (max over mu and sigma), galaxies
    ext_S = ((R['extent'][0] - cx) * 60, (R['extent'][1] - cx) * 60, (R['extent'][2] - cy) * 60, (R['extent'][3] - cy) * 60)
    panels = [(dens(dx, dy), (-half, half, -half, half), 'all stars [arcmin⁻²]', 'Greys', None),
              (dens(dx[sel], dy[sel]), (-half, half, -half, half), f'stars in the CMD mask, μ = {mu} [arcmin⁻²]', 'Greys', None),
              (np.where(R['mask'], R['S'], np.nan), ext_S, 'S, max over μ and σ', 'magma', (0, 8)),
              (dens(gx, gy), (-half, half, -half, half), 'galaxies, r < 21 [arcmin⁻²]', 'Greys', None)]
    for ax, (img, ext, ttl, cmap, lim) in zip(top, panels):
        im = ax.imshow(img.T, origin='lower', extent=ext, cmap=cmap, vmin=lim and lim[0], vmax=lim and lim[1])
        ax.add_patch(plt.Circle((0, 0), r_on, fill=False, color='C0', lw=1))   # the Hess aperture
        ax.set(xlim=(half, -half), ylim=(-half, half), title=ttl, xlabel='Δξ [′]')   # east to the left
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    top[0].set_ylabel('Δη [′]')
    # bright galaxies (r_0 < 17), which the SDSS pipeline may have shredded into star-like pieces: red circles
    b = R['rg'] < 17
    top[3].scatter(gx[b], gy[b], s=60, facecolors='none', edgecolors='C3', lw=1.2, label='galaxy, $r_0$ < 17')
    top[3].legend(fontsize=7, loc='lower left')
    # bottom row: Hess diagrams in 0.1 x 0.5 mag bins
    rr = np.hypot(dx, dy)                                          # star distances from the candidate [arcmin]
    pr = np.hypot(R['PX'] - cx, R['PY'] - cy) * 60                # pixel distances [arcmin]
    on, off = rr < r_on, (rr >= r_off[0]) & (rr < r_off[1])
    # area ratio aperture / annulus; the annulus area counts only observed 1' pixels (it may cross the footprint edge)
    alpha = np.pi * r_on**2 / R['mask'][(pr >= r_off[0]) & (pr < r_off[1])].sum()
    hc, hm = np.arange(-0.4, 1.21, 0.1), np.arange(16, 22.51, 0.5)
    hess = lambda s: np.histogram2d(R['c0'][s], R['g0'][s], [hc, hm])[0]
    H_on, H_off = hess(on), hess(off)
    D = H_on - alpha * H_off                                      # expected 0 everywhere without a stellar system
    exc = sel[on].sum() - alpha * sel[off].sum()                  # excess of masked stars in the aperture
    err = np.sqrt(sel[on].sum() + alpha**2 * sel[off].sum())      # its Poisson error
    vd = max(np.abs(D).max(), 1)
    hpanels = [(H_on, f'aperture R < {r_on:.0f}′: {on.sum()} stars', 'Greys', 0, None),
               (H_off, f'annulus {r_off[0]}′–{r_off[1]}′: {off.sum()} stars', 'Greys', 0, None),
               (D, f'aperture − α × annulus (α = {alpha:.4f})', 'RdBu_r', -vd, vd)]
    for ax, (img, ttl, cmap, v0, v1) in zip(bot, hpanels):
        im = ax.imshow(img.T, origin='lower', extent=(hc[0], hc[-1], hm[0], hm[-1]), aspect='auto', cmap=cmap, vmin=v0, vmax=v1)
        ax.plot(ISO_C, ISO_G + mu, c='C1', lw=1)                  # search isochrone at the candidate's mu
        ax.set(xlim=(-0.4, 1.2), ylim=(22.5, 16), xlabel='$(g-r)_0$', title=ttl)
        plt.colorbar(im, ax=ax, label='stars per bin')
    bot[0].set_ylabel('$g_0$')
    # green contour: the CMD mask of this mu
    bot[2].contour(cc, mm, R['masks'][a].reshape(len(cc), len(mm)).T, [0.5], colors='g', linewidths=1)
    fig.suptitle(f'{label}: S = {R["S"][np.argmin(np.abs(R["PX"][:, 0] - cx)), np.argmin(np.abs(R["PY"][0] - cy))]:.1f}, '
                 f'μ = {mu}, σ = {sig}′; excess of masked stars in the aperture {exc:.1f} ± {err:.1f}')
    plt.show()
    return exc, err


# the holdout region, with its galaxies (SDSS type = 3, cModel r < 21)
gal = load('holdout_sdss_galaxies')
xg, yg = tangent(gal.ra.to_numpy(), gal.dec.to_numpy(), RA0, DEC0)
R_hold = dict(x=x, y=y, c0=c0, g0=g0, w=w, S=S, PX=PX, PY=PY, extent=extent, mask=mask, masks=masks, xg=xg, yg=yg,
              rg=(gal.cmodelmag_r - gal.extinction_r).to_numpy())

### A reference card: Willman 1

First, what does a real satellite look like on a card? Willman 1 lies far from the held-out region (it is one of the
Chapter 5 satellites), so its card reveals nothing about the region. We take its 1.5° patch, apply the cuts of
`holdout_sdss` (−0.3 < g − r < 1.2, g < 22.5) and run the frozen search on it: field CMD, CMD masks and $S$ come from
this patch alone, as for the held-out region.

In [ ]:
# Willman 1: its Chapter 5 patch (1.5 deg radius around the LVDB centre), cut like holdout_sdss, searched with the frozen
# pipeline: field CMD, CMD masks and S computed from this patch alone, exactly as for the holdout region
W1 = (162.3436, 51.0501)                       # patch centre = Willman 1 (LVDB v1.1.1) [deg]
wc = load('willman1_sdss')
wc = wc[(wc.psfmag_g - wc.psfmag_r).between(-0.3, 1.2) & (wc.psfmag_g < 22.5)]   # the cuts of holdout_sdss
wx, wy = tangent(wc.ra.to_numpy(), wc.dec.to_numpy(), *W1)
wg0 = (wc.psfmag_g - wc.extinction_g).to_numpy()
wc0 = wg0 - (wc.psfmag_r - wc.extinction_r).to_numpy()
wext = (-1.5, 1.5, -1.5, 1.5)
wp = np.arange(-1.5, 1.5, pix) + pix / 2                          # pixel centres [deg]
WPX, WPY = np.meshgrid(wp, wp, indexing='ij')
wra, wdec = untangent(WPX, WPY, *W1)
wcov = load('willman1_coverage')
wcells = set(zip(wcov.ira[wcov.n > 20], wcov.idec[wcov.n > 20]))
# footprint: observed coverage cells inside the extraction circle (0.02 deg inside its edge, as in Chapter 5)
wmask = (np.hypot(WPX, WPY) < 1.48) & np.array([(i, j) in wcells for i, j in zip(np.floor(wra.ravel() / 0.1).astype(int),
                                                                             np.floor(wdec.ravel() / 0.1).astype(int))]).reshape(WPX.shape)
Hw = gaussian_filter(np.histogram2d(wc0, wg0, [cb, mb])[0], 1.0)
Bw = np.maximum(Hw, 1.0) / np.maximum(Hw, 1.0).sum()
wmasks = np.array([cmd_mask(satellite_cmd(m), Bw).ravel() for m in MUS])
kw = cmd_index(wc0, wg0)
ww = np.where(kw >= 0, wmasks[:, np.maximum(kw, 0)], False).astype(float)
wS, warg = search_max(wx, wy, ww, wmask, wext)
# the peak within 5' of the catalogued centre, and the (mu, sigma) that gave it
i, j = np.unravel_index(np.argmax(np.where(np.hypot(WPX, WPY) < 5 / 60, wS, 0)), wS.shape)
print(f'Willman 1: S = {wS[i, j]:.1f} at mu = {MUS[warg[i, j, 0]]}, sigma = {SIGMAS[warg[i, j, 1]]}′, '
      f'{np.hypot(WPX[i, j], WPY[i, j]) * 60:.1f}′ from the catalogued centre')
wgal = load('willman1_sdss_galaxies')
wxg, wyg = tangent(wgal.ra.to_numpy(), wgal.dec.to_numpy(), *W1)
R_w1 = dict(x=wx, y=wy, c0=wc0, g0=wg0, w=ww, S=wS, PX=WPX, PY=WPY, extent=wext, mask=wmask, masks=wmasks, xg=wxg, yg=wyg,
            rg=(wgal.cmodelmag_r - wgal.extinction_r).to_numpy())
w1_card = report_card(R_w1, WPX[i, j], WPY[i, j], warg[i, j, 0], SIGMAS[warg[i, j, 1]], 'Willman 1 (reference)')

**Figure 9.2.** Report card of Willman 1 — a real satellite at high S/N ($M_V = -2.5$, μ = 17.9, $r_h = 2.5′$ in LVDB
v1.1.1): $S$ = 14.9 at μ = 17.5 (the nearest grid value) with the σ = 2′ kernel, 0.7′ from the catalogued centre
($S$ = 15.4 in Chapter 5, with the deeper g < 23 sample). Panels as described in Section 9.5; aperture R < 4′, annulus
30′–60′, α = 0.0060.

*Reading.* The overdensity is resolved, ~4′ across, and visible among all stars as well as in the CMD mask; $S$ is a
single isolated peak; the galaxy map is featureless at the position, with no bright galaxy. The aperture holds 73 stars,
40.6 ± 6.6 more masked stars than the annulus predicts, and the difference Hess puts them on the isochrone: a main
sequence and turn-off at $g_0 \approx 21$–22.5, $(g-r)_0 \approx 0.1$–0.4. (A satellite this faint has almost no giants
in SDSS.) This is the pattern to look for: a resolved overdensity whose stars follow one isochrone at one distance.

In [ ]:
cards = {}
for n, c in cand.head(4).iterrows():            # the four strongest candidates
    cx, cy = tangent(c.ra, c.dec, RA0, DEC0)                      # candidate position in the tangent plane [deg]
    cards[n] = report_card(R_hold, cx, cy, list(MUS).index(c.mu), c.sigma_arcmin, f'candidate #{n}')

**Figure 9.3.** Report cards of the four strongest candidates of the held-out region, #0–#3 ($S$ = 13.7, 10.0, 9.1 and
8.0), all found with the 1′ kernel, at μ = 17.5, 21.0, 21.0 and 21.5. Panels as in Figure 9.2; aperture R < 3′ for all
four (α ≈ 0.0033). In the S maps of #2 and #3, the black band at the bottom is outside the searched area (the southern
edge of the region at Dec = −3°). Excess of masked stars in the aperture: 25.6 ± 5.3, 11.3 ± 3.6, 9.5 ± 3.3 and
10.6 ± 3.5.

*Reading.*
- **#0** has an $S$ close to Willman 1's, but a different card. The stars form an unresolved knot (≲ 1′, i.e. ≲ 10 pc
  at μ = 17.5); the excess lies at $g_0 > 21$ with $(g-r)_0 \approx -0.3$ to 0.3 — mostly bluer than the turn-off of the
  isochrone — with no subgiants or giants; and two bright galaxies ($r_0 < 17$) lie within 0.3′ of the peak. A large
  galaxy split by the pipeline into star-like pieces explains all three. Verdict: not a satellite.
- **#1, #2, #3** are knots of ≲ 1′ at μ = 21–21.5, where the mask selects mainly the horizontal-branch region. Each
  excess is ~10 sources in the faintest rows of the Hess diagram, scattered in colour, with no giant branch. No bright
  galaxy sits at any of them; #2 and #3 lie a few arcminutes from galaxy overdensities (#2: 5′ south; #3: 8′ north-west).
  At this depth the card cannot exclude a compact distant system, but nothing on it looks like a stellar population.
  Verdict: unconfirmed; they need deeper data before any claim.

## 9.6 Unblinding: what is known in this region?

Only now do we look at the Local Volume Database. The region contains three Milky Way satellites — Leo IV
($M_V = -4.95$, 151 kpc), Leo V ($M_V = -4.4$, 168 kpc) and Virgo I ($M_V = -0.9$, 91 kpc, discovered in the much
deeper HSC data) — and the galaxy group around NGC 3521 at ~11 Mpc, whose dwarfs are background galaxies.

In [ ]:
lvdb = load('lvdb').set_index('key')            # opened only now, after the search and its threshold are frozen
known = lvdb.loc[['leo_4', 'leo_5', 'virgo_1']]                   # the known Milky Way satellites in the region
for key, k in known.iterrows():
    # angular distance [arcmin] of every candidate from the satellite (flat sky, RA difference scaled by cos dec)
    d = np.hypot((cand.ra - k.ra) * np.cos(np.radians(k.dec)), cand.dec - k.dec) * 60
    hit = d.min() < max(3, k.rhalf) if len(cand) else False       # matched within max(3', r_h) (rhalf in arcmin)
    print(f'{key:8s} M_V = {k.M_V:5.2f}, mu = {k.distance_modulus:.2f}: '
          + (f'recovered as candidate #{d.idxmin()} ({d.min():.1f}′ away, S = {cand.S[d.idxmin()]:.1f}, mu = {cand.mu[d.idxmin()]})'
             if hit else 'not recovered'))
# Test of this chapter: Leo IV and Leo V, which Chapter 8 says SDSS should find (M_V <~ -4 at 150-170 kpc), are each
# within 3' of a candidate (measured offsets ~1'). Virgo I, known only from deeper data, is not required.
for key in ('leo_4', 'leo_5'):
    k = known.loc[key]
    assert (np.hypot((cand.ra - k.ra) * np.cos(np.radians(k.dec)), cand.dec - k.dec) * 60 < 3).any(), key

Both SDSS satellites of the region are recovered, at the right distance — but only at S ≈ 6.5, below many other
candidates. What are those? Their positions were looked up in SIMBAD (https://simbad.cds.unistra.fr; a 2′ cone search
for galaxies, galaxy groups, clusters, quasars and star clusters, done by the authors with the copy of SIMBAD in WSDB):

| Candidate | SIMBAD object within 2′ | Type |
|---|---|---|
| #0 (S = 13.7) | UGC 6903, 0.3′ | emission-line (star-forming) galaxy |
| #3 | UGC 6780, 0.2′ | emission-line galaxy |
| #5 | SDSS J110601.01+042514.7, 0.6′ | quasar |
| #8 | NGC 3664, 0.6′; WBL 329, 1.3′ | interacting galaxy; group of galaxies |
| #12 | [VV2006] J113245.8−004231, 1.9′ | quasar |
| others | none of these types | — |

The strongest "satellite" of the region is a nearby star-forming galaxy. The SDSS pipeline splits ("shreds") a large
galaxy into many pieces — star-forming knots, HII regions — and many of them are classified as point sources with blue
colours that fall in our CMD mask (Figure 9.3, #0: an unresolved knot of blue sources on two bright galaxy pieces). Most other
candidates are found with the smallest kernel (1′) at the largest distances (μ = 21–21.5), where the mask selects faint,
blue point sources: exactly the regime where half of the SDSS "stars" are galaxies (Chapter 1), and galaxies cluster.

**Our null was too optimistic.** It placed stars uniformly; the real sky has galaxy groups, shredded galaxies and
clustered faint galaxies, so the real false-positive rate is far higher than 1/30 per region. Real searches mask
catalogued bright galaxies before searching, vet every candidate with star–galaxy tests (Chapter 3) and deeper data, and
calibrate thresholds on real fields — or, equivalently, report the candidates with their vetting, not just their S.

In [ ]:
# Leo IV: the candidate matched to it above
k = known.loc['leo_4']
n4 = (np.hypot((cand.ra - k.ra) * np.cos(np.radians(k.dec)), cand.dec - k.dec)).idxmin()
c = cand.loc[n4]
cx, cy = tangent(c.ra, c.dec, RA0, DEC0)
leo4_card = report_card(R_hold, cx, cy, list(MUS).index(c.mu), c.sigma_arcmin, f'Leo IV = candidate #{n4}')

**Figure 9.4.** Report card of Leo IV — a real satellite at low S/N ($M_V = -5.0$, μ = 20.9, $r_h = 2.5′$ in LVDB
v1.1.1), candidate #10: $S$ = 6.5 at μ = 21.0 with the σ = 4′ kernel. Panels as in Figure 9.2; aperture R < 8′,
α = 0.0238.

*Reading.* Leo IV has half the $S$ of candidate #0, yet a better card. The masked stars form an irregular, resolved
overdensity ~10′ across (in the all-star map it is barely visible: the mask is what reveals it). The excess, 18.9 ± 5.5
masked stars, lies on one isochrone: red giants at $(g-r)_0 \approx 0.5$–0.6, $g_0 \approx 21$ (the strongest bin) and blue
horizontal-branch stars at $g_0 \approx 21.5$, $(g-r)_0 \approx -0.2$ to 0. The galaxy map has a small knot ~2′
south-west of the centre and a few scattered bright galaxies — a reminder that galaxies are clustered everywhere and
cannot by themselves veto a candidate.

**The lesson of the cards.** Ranking by $S$ put the two real satellites of the region at #9 and #10, below a shredded
galaxy and a dozen unresolved knots. The cards rank them differently: what makes a satellite is not the height of the
peak but a resolved overdensity whose stars follow one isochrone — with a giant branch and horizontal branch at the
same distance — and that is not explained by a galaxy. Where SDSS is too shallow to show that (μ ≳ 21), the honest
verdict is "unconfirmed" until deeper data are in hand.

Compare the outcome with Chapter 8: at 150–170 kpc the SDSS search should find satellites with $M_V \lesssim -4$, and
miss ones as faint as Virgo I — as it does. Any candidate not matched to a known object deserves the Chapter 3
treatment (is the excess made of stars?) and a deeper look (Legacy Surveys, HSC) before it is called new.

**Exercise.** Mask circles of 2′ around the SIMBAD galaxies above and rerun the search; then apply the colour test of
Chapter 3 to the remaining candidates.

## Summary

- A frozen pipeline, a footprint mask and a threshold from null realisations of the same region make a blind search.
- Report cards (maps of all stars, masked stars, S and galaxies; on, off and difference Hess diagrams) let a human
  judge each detection: a satellite is a resolved overdensity that follows one isochrone and is not a galaxy.
- The known SDSS satellites in the region (Leo IV, Leo V) are recovered; Virgo I, found only in deeper data, is not.
- A null with uniformly placed stars underestimates false positives: shredded nearby galaxies and clustered faint
  galaxies dominate the candidate list. Vetting is part of the search.